In [ ]:
# AI Categorization: Accuracy & Performance Analytics

**Objective:**
This notebook evaluates the performance of the fine-tuned Gemma 4 (26B) categorization model. It compares the AI-generated predictions against the verified ground-truth data (from the Mercari dataset) to calculate hierarchical accuracy, precision, recall, and F1-scores.

**Instructions:**
Ensure you point the `GENERATED_FILE_PATH` to the correct output from the previous inference script. You must specify whether you are evaluating the **10% Unseen Test Set** (recommended for true model evaluation) or the **100% Full Dataset** (for mass unification analytics).

In [ ]:
# Install required analytics libraries
!pip install -q pandas matplotlib scikit-learn openpyxl

import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

# --- Configuration & File Paths ---

# 1. Ground Truth File (The original verified Mercari data)
TRUTH_FILE_PATH = "path/to/your/output_directory/Split_Files/Categorized.xlsx"

# 2. AI Generated Predictions File
# INSTRUCTION: Choose the correct file based on what you generated in Script 06 (10% vs 100%)
# Option A (10% Test Set Output): "path/to/your/aws_outputs/Generated_10_Percent.xlsx"
# Option B (100% Full Data Output): "path/to/your/aws_outputs/Generated_Categorization_Data.xlsx"
GENERATED_FILE_PATH = "path/to/your/aws_outputs/Generated_Categorization_Data.xlsx"

# Column Definitions
TITLE_COL = "item_name"
TRUE_COLS = ["category_1", "category_2", "category_3"]
PRED_COLS = ["category_1_pred", "category_2_pred", "category_3_pred"]

print("Dependencies loaded and configuration set.")

In [ ]:
# --- Data Loading & Preprocessing ---
print("Loading ground-truth and prediction datasets...")

try:
    df_truth = pd.read_excel(TRUTH_FILE_PATH)
    df_pred = pd.read_excel(GENERATED_FILE_PATH)
except FileNotFoundError as e:
    raise FileNotFoundError(f"❌ Error loading files. Please check your paths: {e}")

# Isolate only the necessary columns to optimize memory usage
df_truth = df_truth[[TITLE_COL] + TRUE_COLS].copy()
df_pred = df_pred[[TITLE_COL] + PRED_COLS].copy()

# Fill missing values with a string placeholder to ensure safe mathematical comparisons
df_truth.fillna("None", inplace=True)
df_pred.fillna("None", inplace=True)

# --- Merge Datasets ---
# We use an inner join on the item name to align the true categories with the predicted ones
df_merged = pd.merge(df_truth, df_pred, on=TITLE_COL, how="inner")
total_rows = len(df_merged)

if total_rows == 0:
    raise ValueError("❌ No matching items were found between the two files. Please verify the datasets.")

print(f"✅ Successfully matched and merged {total_rows} products for analysis.\n")

In [ ]:
# --- Calculate Custom Hierarchical Metrics ---

# 1. Full Match (Perfect Classification across all 3 levels)
full_match = (
    (df_merged[TRUE_COLS[0]] == df_merged[PRED_COLS[0]]) & 
    (df_merged[TRUE_COLS[1]] == df_merged[PRED_COLS[1]]) & 
    (df_merged[TRUE_COLS[2]] == df_merged[PRED_COLS[2]])
)

# 2. Level 1 Correct (Overall Macro-category success)
cat1_correct = (df_merged[TRUE_COLS[0]] == df_merged[PRED_COLS[0]])

# 3. Partial Failure: Level 1 Correct, but Level 2 Wrong
cat1_correct_cat2_wrong = (
    (df_merged[TRUE_COLS[0]] == df_merged[PRED_COLS[0]]) & 
    (df_merged[TRUE_COLS[1]] != df_merged[PRED_COLS[1]])
)

# 4. Partial Failure: Level 1 & 2 Correct, but Level 3 Wrong
cat1_cat2_correct_cat3_wrong = (
    (df_merged[TRUE_COLS[0]] == df_merged[PRED_COLS[0]]) & 
    (df_merged[TRUE_COLS[1]] == df_merged[PRED_COLS[1]]) & 
    (df_merged[TRUE_COLS[2]] != df_merged[PRED_COLS[2]])
)

# --- Calculate Percentages ---
pct_full = (full_match.sum() / total_rows) * 100
pct_cat1 = (cat1_correct.sum() / total_rows) * 100
pct_c1_c2wrong = (cat1_correct_cat2_wrong.sum() / total_rows) * 100
pct_c1c2_c3wrong = (cat1_cat2_correct_cat3_wrong.sum() / total_rows) * 100

print("="*60)
print("📊 HIERARCHICAL ACCURACY BREAKDOWN")
print("="*60)
print(f"1. Full Accuracy (L1, L2 & L3 Correct) : {pct_full:.2f}% ({full_match.sum()} items)")
print(f"2. Cat 1 Base Accuracy (L1 Success)    : {pct_cat1:.2f}% ({cat1_correct.sum()} items)")
print(f"3. L1 Correct, but L2 Wrong            : {pct_c1_c2wrong:.2f}% ({cat1_correct_cat2_wrong.sum()} items)")
print(f"4. L1 & L2 Correct, but L3 Wrong       : {pct_c1c2_c3wrong:.2f}% ({cat1_cat2_correct_cat3_wrong.sum()} items)")
print("="*60)

In [ ]:
# --- Generate Standard ML Metrics (Precision, Recall, F1) ---
print("\nGenerating Statistical Report (This may take a moment for large datasets)...")

report_l1 = classification_report(df_merged[TRUE_COLS[0]], df_merged[PRED_COLS[0]], zero_division=0, output_dict=True)
report_l2 = classification_report(df_merged[TRUE_COLS[1]], df_merged[PRED_COLS[1]], zero_division=0, output_dict=True)
report_l3 = classification_report(df_merged[TRUE_COLS[2]], df_merged[PRED_COLS[2]], zero_division=0, output_dict=True)

# Weighted average accounts for class imbalances (some categories having more items)
avg_type = 'weighted avg' 

metrics_data = {
    'Taxonomy Level': ['Level 1 (Macro)', 'Level 2 (Mid)', 'Level 3 (Micro)'],
    'Precision': [f"{report_l1[avg_type]['precision']:.3f}", f"{report_l2[avg_type]['precision']:.3f}", f"{report_l3[avg_type]['precision']:.3f}"],
    'Recall':    [f"{report_l1[avg_type]['recall']:.3f}",    f"{report_l2[avg_type]['recall']:.3f}",    f"{report_l3[avg_type]['recall']:.3f}"],
    'F1-Score':  [f"{report_l1[avg_type]['f1-score']:.3f}",  f"{report_l2[avg_type]['f1-score']:.3f}",  f"{report_l3[avg_type]['f1-score']:.3f}"]
}

metrics_df = pd.DataFrame(metrics_data)
print("\n" + "="*60)
print("📈 AVERAGE METRICS SUMMARY TABLE (WEIGHTED)")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60 + "\n")

# --- Generate Visualization ---
labels = ['Perfect Match\n(All 3 Levels)', 'Level 1 Base\nAccuracy', 'L1 Correct\nL2 Wrong', 'L1+L2 Correct\nL3 Wrong']
values = [pct_full, pct_cat1, pct_c1_c2wrong, pct_c1c2_c3wrong]
colors = ['#2E7D32', '#1565C0', '#F9A825', '#C62828'] # Enterprise-friendly colors

plt.figure(figsize=(10, 6))
bars = plt.bar(labels, values, color=colors, edgecolor='black', alpha=0.85)

# Styling the chart
plt.title('AI Categorization Accuracy & Error Breakdown', fontsize=15, fontweight='bold', pad=15)
plt.ylabel('Percentage of Total Products (%)', fontsize=12, fontweight='bold')
plt.ylim(0, max(values) + 15) 
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Annotate bars with data labels
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.1f}%", ha='center', va='bottom', fontsize=12, fontweight='bold')

# Remove top and right spines for a cleaner look
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()
plt.show()